# 语言模型零基础 05：从语料到 ARPA 与 G.fst

这一课把前四课串成真实语言模型流水线：

```text
分词语料 → KenLM lmplz → ARPA → kaldilm/arpa2fst → OpenFst G.fst
```

完成本课后，你应该能够：

1. 使用 Modified Kneser–Ney 训练 1/2/3-gram；
2. 读懂 ARPA 的 log10 probability 与 backoff weight；
3. 手工执行一次 backoff 概率查询并计算 PPL；
4. 把 ARPA 转成 OpenFst `G.fst`；
5. 解释 epsilon backoff 与 Kaldi `#0` backoff 的区别；
6. 验证 ARPA 句子概率与 OpenFst 路径代价一致。


## 本课使用的真实工具

- KenLM `lmplz`：从文本估计 Modified Kneser–Ney N-gram，输出 ARPA；
- KenLM `build_binary/query`：构建高效二进制模型并查询句子；
- `kaldilm`：Kaldi `arpa2fst` 的 Python wrapper；
- OpenFst：检查、打印、组合和最短路径。

> 小语料不足以稳定估计折扣，本课使用 `--discount_fallback` 只是为了让教学实验可运行。生产训练应使用足够大、清洗和去重后的语料，并在验证集上调参与剪枝。


In [ ]:
from pathlib import Path
from math import isclose, log, log10
from collections import defaultdict
import subprocess

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_project_root()
LAB = ROOT / "openfst_lab" / "lesson05"
LAB.mkdir(parents=True, exist_ok=True)

LMPLZ = "/opt/kenlm/build/bin/lmplz"
BUILD_BINARY = "/opt/kenlm/build/bin/build_binary"
QUERY = "/opt/kenlm/build/bin/query"
KALDILM_PYTHON = "/opt/kaldilm-venv/bin/python"

def run_wsl(*args, input_text=None, check=True):
    result = subprocess.run(
        ["wsl", "-d", "Ubuntu", "--", *map(str, args)],
        input=input_text, text=True, capture_output=True, check=False,
        encoding="utf-8", errors="replace",
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"命令失败：{args}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}")
    return result

def to_wsl_path(path):
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(":").lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f"/mnt/{drive}/{relative}"

def write_lf(path, text):
    Path(path).write_text(text, encoding="utf-8", newline="\n")

for tool in [LMPLZ, BUILD_BINARY, QUERY, KALDILM_PYTHON]:
    result = run_wsl("test", "-x", tool, check=False)
    print(tool, "OK" if result.returncode == 0 else "MISSING")
    assert result.returncode == 0
print("实验目录：", LAB)


## 1. 准备训练、验证和测试文本

每一行是一句话，词之间使用空格。真实项目必须先固定 normalization、分词和词表；训练、验证、测试必须分离。


In [ ]:
train_sentences = [
    "jintian tianqi henhao", "jintian tianqi bucuo",
    "jintian tianqi hen leng", "jintian tianqi hen re",
    "mingtian tianqi henhao", "mingtian tianqi bucuo",
    "zuotian tianqi bucuo", "zuotian tianqi hen leng",
    "jintian xinqing henhao", "jintian xinqing bucuo",
    "mingtian xinqing henhao", "zuotian xinqing bucuo",
    "wo xihuan yuyin shibie", "wo xuexi yuyin shibie",
    "ni xihuan yuyan moxing", "ni xuexi yuyan moxing",
    "wo xihuan ziran yuyan", "wo xuexi ziran yuyan",
    "ni xihuan ziran yuyan", "ni xuexi yuyin shibie",
    "yuyin shibie xuyao yuyan moxing",
    "ziran yuyan keyi bangzhu shibie",
    "jintian women xuexi openfst",
    "mingtian women xuexi ngram",
]
dev_sentences = [
    "jintian tianqi bucuo",
    "wo xuexi yuyan moxing",
    "yuyin shibie xuyao ngram",
]
test_sentences = [
    "mingtian tianqi hen leng",
    "ni xihuan yuyin shibie",
]

train_path = LAB / "train.txt"
write_lf(train_path, "\n".join(train_sentences) + "\n")
print("训练句数：", len(train_sentences))
print("训练 token 数：", sum(len(s.split()) for s in train_sentences))
print(train_path.read_text(encoding="utf-8")[:500])


## 2. 用 KenLM 训练 Trigram ARPA

`lmplz -o 3` 训练最高阶为 3 的模型。KenLM 输出 Modified Kneser–Ney 平滑后的 ARPA。ARPA 中概率和 backoff weight 使用以 10 为底的对数。


In [ ]:
arpa_path = LAB / "tiny.3gram.arpa"
train_result = run_wsl(
    LMPLZ, "--order", "3",
    "--text", to_wsl_path(train_path),
    "--arpa", to_wsl_path(arpa_path),
    "--discount_fallback",
    "--memory", "256M",
)
print("lmplz return code：", train_result.returncode)
print("ARPA bytes：", arpa_path.stat().st_size)
print("训练日志末尾：")
print("\n".join(train_result.stderr.splitlines()[-12:]))
assert arpa_path.exists() and arpa_path.stat().st_size > 0


## 3. ARPA 文件结构

典型结构：

```text
\data\
ngram 1=...
ngram 2=...
ngram 3=...

\1-grams:
log10_probability    word             backoff_log10

\2-grams:
log10_probability    word1 word2       backoff_log10

\3-grams:
log10_probability    word1 word2 word3
```

最高阶通常没有 backoff weight。某个高阶 N-gram 缺失时，先加当前 history 的 backoff weight，再查询低一阶。


In [ ]:
arpa_text = arpa_path.read_text(encoding="utf-8")
print("\n".join(arpa_text.splitlines()[:45]))


In [ ]:
def parse_arpa(text):
    entries = defaultdict(dict)
    declared_counts = {}
    current_order = None
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if line.startswith("ngram "):
            left, count = line.split("=")
            declared_counts[int(left.split()[1])] = int(count)
        elif line.startswith("\\") and line.endswith("-grams:"):
            current_order = int(line[1:].split("-")[0])
        elif line == "\\end\\":
            current_order = None
        elif current_order and line and not line.startswith("\\"):
            fields = line.split()
            log10_probability = float(fields[0])
            ngram = tuple(fields[1:1 + current_order])
            backoff = float(fields[1 + current_order]) if len(fields) > 1 + current_order else 0.0
            entries[current_order][ngram] = (log10_probability, backoff)
    return dict(entries), declared_counts

arpa_entries, arpa_counts = parse_arpa(arpa_text)
print("ARPA 声明数量：", arpa_counts)
print("实际解析数量：", {order: len(rows) for order, rows in arpa_entries.items()})
assert arpa_counts == {order: len(rows) for order, rows in arpa_entries.items()}


## 4. 手工执行 backoff 查询

查询 `P(word | history)` 时，从最高可用阶开始：

1. 如果完整 N-gram 存在，直接使用它的 log10 probability；
2. 如果不存在，加上当前 history 的 backoff weight；
3. 删除最老的 history token，继续查低一阶；
4. 最终回退到 unigram。


In [ ]:
MAX_ORDER = max(arpa_entries)

def arpa_word_log10(word, history, trace=False):
    history = tuple(history[-(MAX_ORDER - 1):])
    accumulated_backoff = 0.0
    steps = []
    for history_length in range(len(history), -1, -1):
        suffix = history[-history_length:] if history_length else ()
        ngram = suffix + (word,)
        order = len(ngram)
        if ngram in arpa_entries.get(order, {}):
            direct_log10, _ = arpa_entries[order][ngram]
            steps.append(("hit", ngram, direct_log10))
            result = accumulated_backoff + direct_log10
            if trace:
                for step in steps:
                    print(step)
                print("total log10=", result)
            return result
        if history_length > 0:
            history_entry = arpa_entries.get(history_length, {}).get(suffix)
            backoff = history_entry[1] if history_entry else 0.0
            accumulated_backoff += backoff
            steps.append(("backoff", suffix, backoff))
    raise KeyError(f"unigram 中也没有 {word!r}，应先映射为 <unk>")

print("已见上下文：")
arpa_word_log10("henhao", ["jintian", "tianqi"], trace=True)
print("\n需要回退的上下文：")
arpa_word_log10("bucuo", ["wo", "xuexi"], trace=True)


In [ ]:
def normalize_oov(tokens):
    unigram_vocab = {ngram[0] for ngram in arpa_entries[1]}
    return [word if word in unigram_vocab else "<unk>" for word in tokens]

def sentence_log10_probability(sentence, verbose=False):
    words = normalize_oov(sentence.split())
    history = ["<s>"]
    total = 0.0
    for word in [*words, "</s>"]:
        value = arpa_word_log10(word, history)
        if verbose:
            print(f"log10 P({word} | {' '.join(history[-2:])}) = {value:.6f}")
        total += value
        history.append(word)
        history = history[-(MAX_ORDER - 1):]
    return total

def perplexity(sentences):
    total_log10 = sum(sentence_log10_probability(sentence) for sentence in sentences)
    predicted = sum(len(sentence.split()) + 1 for sentence in sentences)  # 包含 </s>，不含 <s>
    return 10 ** (-total_log10 / predicted)

example = "jintian tianqi henhao"
manual_log10 = sentence_log10_probability(example, verbose=True)
print("sentence log10 probability：", manual_log10)
print("sentence probability：", 10 ** manual_log10)
print("dev PPL：", perplexity(dev_sentences))


## 5. 比较不同阶数

阶数更高不保证在小数据上更好。下面分别训练 1/2/3-gram，在同一个 validation set 上比较 PPL。测试集不能用于选阶数。


In [ ]:
def train_arpa(order):
    output = LAB / f"tiny.{order}gram.arpa"
    run_wsl(
        LMPLZ, "--order", str(order), "--text", to_wsl_path(train_path),
        "--arpa", to_wsl_path(output), "--discount_fallback", "--memory", "256M",
    )
    return output

def perplexity_from_arpa(path, sentences):
    global arpa_entries, MAX_ORDER
    old_entries, old_order = arpa_entries, MAX_ORDER
    try:
        arpa_entries, _ = parse_arpa(Path(path).read_text(encoding="utf-8"))
        MAX_ORDER = max(arpa_entries)
        return perplexity(sentences)
    finally:
        arpa_entries, MAX_ORDER = old_entries, old_order

order_results = []
for order in [1, 2, 3]:
    model_path = train_arpa(order)
    order_results.append((order, perplexity_from_arpa(model_path, dev_sentences)))
for order, ppl in order_results:
    print(f"{order}-gram dev PPL = {ppl:.4f}")
best_order = min(order_results, key=lambda item: item[1])[0]
print("这个小验证集上的最佳阶数：", best_order)


## 6. 构建 KenLM 二进制并交叉检查

ARPA 便于查看和交换，但线上查询常使用压缩二进制结构。`build_binary` 不改变模型概率，只改变存储和查询方式。


In [ ]:
kenlm_binary = LAB / "tiny.3gram.klm"
run_wsl(BUILD_BINARY, to_wsl_path(arpa_path), to_wsl_path(kenlm_binary))
query_input = example + "\n"
query_arpa = run_wsl(QUERY, to_wsl_path(arpa_path), input_text=query_input).stdout
query_binary = run_wsl(QUERY, to_wsl_path(kenlm_binary), input_text=query_input).stdout
print("ARPA query：\n", query_arpa)
print("Binary query：\n", query_binary)
print("二进制模型 bytes：", kenlm_binary.stat().st_size)


## 7. 建立 word symbol table

OpenFst 内部标签是整数。`L.output` 与 `G.input` 必须共享同一个 word symbol table。Kaldi 风格 `G` 还需要 `#0` 作为输入侧 backoff/disambiguation label。


In [ ]:
unigram_vocab = {ngram[0] for ngram in arpa_entries[1]}
special_order = ["<eps>", "<s>", "</s>", "<unk>"]
normal_words = sorted(unigram_vocab - set(special_order))
symbols = [*special_order, *normal_words, "#0"]
words_path = LAB / "words.txt"
write_lf(words_path, "\n".join(f"{word} {index}" for index, word in enumerate(symbols)) + "\n")
symbol_to_id = {word: index for index, word in enumerate(symbols)}
print(words_path.read_text(encoding="utf-8"))
assert symbol_to_id["<eps>"] == 0
assert "<unk>" in symbol_to_id and "#0" in symbol_to_id


## 8. ARPA → 两种 G.fst

我们生成两份图：

- `G.eps.fst`：backoff 使用 epsilon，便于本课直接和线性句子图组合、验证概率；
- `G.kaldi.fst`：backoff 输入侧使用 `#0`、输出侧使用 epsilon，适合后续 Kaldi 风格 `L∘G` 图构建。

第二份严格说不是纯 acceptor，因为 `#0:ε` 弧的输入输出不同；除这些 backoff 弧外，普通 word 弧输入输出相同。


In [ ]:
g_eps = LAB / "G.eps.fst"
g_kaldi = LAB / "G.kaldi.fst"

common_kaldilm = [
    KALDILM_PYTHON, "-m", "kaldilm",
    f"--read-symbol-table={to_wsl_path(words_path)}",
    "--keep-symbols=true",
]
run_wsl(*common_kaldilm, to_wsl_path(arpa_path), to_wsl_path(g_eps))
run_wsl(
    *common_kaldilm, "--disambig-symbol=#0",
    to_wsl_path(arpa_path), to_wsl_path(g_kaldi),
)

for name, path in [("G.eps", g_eps), ("G.kaldi", g_kaldi)]:
    info = run_wsl("fstinfo", to_wsl_path(path)).stdout
    selected = [line.strip() for line in info.splitlines() if line.strip().startswith((
        "# of states", "# of arcs", "# of input epsilons", "acceptor", "input deterministic"
    ))]
    print(name)
    print("\n".join(selected))
    print()


In [ ]:
print("G.kaldi 前 30 行：")
g_printed = run_wsl(
    "fstprint", f"--isymbols={to_wsl_path(words_path)}",
    f"--osymbols={to_wsl_path(words_path)}", to_wsl_path(g_kaldi)
).stdout
print("\n".join(g_printed.splitlines()[:30]))
backoff_lines = [line for line in g_printed.splitlines() if "#0" in line]
print("\n#0 backoff 弧数量：", len(backoff_lines))
print("示例：", backoff_lines[:3])
assert backoff_lines


## 9. 验证 ARPA 概率与 OpenFst 路径代价

ARPA 保存 `log10(P)`；OpenFst standard/tropical 图保存自然对数域代价：

$$cost=-\ln P=-\log_{10}(P)\ln(10)$$

下面为每个句子构造线性 acceptor，与 `G.eps` 组合后取最短路径并累加权重。结果应与手算 ARPA 成本一致。


In [ ]:
def make_sentence_acceptor(sentence, stem):
    # kaldilm 生成的 G 显式读取句首/句尾标记，所以线性 acceptor 也要带上它们。
    words = ["<s>", *normalize_oov(sentence.split()), "</s>"]
    text_path = LAB / f"{stem}.txt"
    fst_path = LAB / f"{stem}.fst"
    lines = [f"{i} {i+1} {word} 0" for i, word in enumerate(words)]
    lines.append(str(len(words)))
    write_lf(text_path, "\n".join(lines) + "\n")
    run_wsl(
        "fstcompile", "--acceptor=true",
        f"--isymbols={to_wsl_path(words_path)}", f"--osymbols={to_wsl_path(words_path)}",
        "--keep_isymbols=true", "--keep_osymbols=true",
        to_wsl_path(text_path), to_wsl_path(fst_path),
    )
    return fst_path

def printed_path_cost(text):
    total = 0.0
    for line in text.splitlines():
        fields = line.split()
        if len(fields) >= 5:
            total += float(fields[4])
        elif len(fields) == 2:
            total += float(fields[1])
    return total

def sentence_fst_cost(sentence, stem):
    sentence_fst = make_sentence_acceptor(sentence, stem)
    sorted_fst = LAB / f"{stem}.sorted.fst"
    composed = LAB / f"{stem}.with_G.fst"
    best = LAB / f"{stem}.best.fst"
    run_wsl("fstarcsort", "--sort_type=olabel", to_wsl_path(sentence_fst), to_wsl_path(sorted_fst))
    run_wsl("fstcompose", to_wsl_path(sorted_fst), to_wsl_path(g_eps), to_wsl_path(composed))
    states = int(next(line.split()[-1] for line in run_wsl('fstinfo', to_wsl_path(composed)).stdout.splitlines() if line.strip().startswith('# of states')))
    if states == 0:
        raise RuntimeError(f"句子与 G 组合后为空：{sentence}")
    run_wsl("fstshortestpath", to_wsl_path(composed), to_wsl_path(best))
    printed = run_wsl("fstprint", to_wsl_path(best)).stdout
    return printed_path_cost(printed)

comparison_sentences = [
    "jintian tianqi henhao",
    "jintian xinqing bucuo",
    "wo xuexi yuyan moxing",
]
for index, sentence in enumerate(comparison_sentences):
    arpa_cost = -sentence_log10_probability(sentence) * log(10)
    fst_cost = sentence_fst_cost(sentence, f"sentence_{index}")
    difference = abs(arpa_cost - fst_cost)
    print(f"{sentence:28s} ARPA={arpa_cost:.6f} FST={fst_cost:.6f} diff={difference:.2e}")
    assert difference < 1e-4


## 10. ASR 中怎样融合 acoustic score 与 LM score

实际解码不会只看语言模型。常见形式是：

```text
total_cost = acoustic_cost + lm_weight * lm_cost + insertion_penalty * word_count
```

所有权重必须在 validation set 上调，不能在 test set 上挑最优值。LM 权重太大会让系统忽略声音、偏向常见句。


In [ ]:
import ipywidgets as widgets

hypotheses = [
    {"text": "jintian tianqi henhao", "acoustic_cost": 3.0},
    {"text": "jintian xinqing henhao", "acoustic_cost": 2.6},
]

def fusion_demo(lm_weight=0.5, insertion_penalty=0.0):
    scored = []
    for hypothesis in hypotheses:
        lm_cost = -sentence_log10_probability(hypothesis["text"]) * log(10)
        total = (hypothesis["acoustic_cost"] + lm_weight * lm_cost
                 + insertion_penalty * len(hypothesis["text"].split()))
        scored.append((total, hypothesis["text"], hypothesis["acoustic_cost"], lm_cost))
    for total, text, acoustic, lm_cost in sorted(scored):
        print(f"total={total:.3f} acoustic={acoustic:.3f} LM={lm_cost:.3f}  {text}")
    print("最终选择：", min(scored)[1])

widgets.interact(
    fusion_demo,
    lm_weight=widgets.FloatSlider(value=0.5, min=0, max=2, step=0.1),
    insertion_penalty=widgets.FloatSlider(value=0.0, min=-1, max=1, step=0.1),
);
fusion_demo(0.5, 0.0)


## 11. 生产环境还需要什么？

- 大规模、去重、领域匹配的文本；
- 固定 normalization、tokenization 和 OOV 策略；
- 在 validation set 上选择 order、剪枝阈值、LM weight 和 insertion penalty；
- 检查 `G.fst` 的 stochasticity、可达性、确定性和 symbol table；
- 构造与 `#0` 匹配的词典 disambiguation loop 后再做 `L∘G`；
- 分别评测 WER/CER、实体错误率、延迟、内存和领域外退化。


## 12. 自动判题


In [ ]:
# 请修改八个答案
answer_1 = ""  # KenLM ARPA 概率使用 log10 还是 ln？
answer_2 = ""  # OpenFst 中从概率得到代价：-log10(p) 还是 -ln(p)？
answer_3 = ""  # 高阶 N-gram 缺失时使用什么机制？英文
answer_4 = ""  # 词表外词通常映射到哪个 token？
answer_5 = ""  # Kaldi 风格 G 的 backoff 输入标签
answer_6 = ""  # PPL 在相同评测设置下越 lower 还是 higher 越好？
answer_7 = ""  # LM weight 应在 validation 还是 test set 调？
answer_8 = ""  # ARPA 转 OpenFst G 的 Kaldi 命令名（kaldilm 包装了它）

checks = [
    str(answer_1).strip().lower() == "log10",
    str(answer_2).strip().lower() == "-ln(p)",
    str(answer_3).strip().lower() == "backoff",
    str(answer_4).strip().lower() == "<unk>",
    str(answer_5).strip() == "#0",
    str(answer_6).strip().lower() == "lower",
    str(answer_7).strip().lower() == "validation",
    str(answer_8).strip().lower() == "arpa2fst",
]
for number, ok in enumerate(checks, 1):
    print(("✅" if ok else "❌"), f"第 {number} 题")
print(f"得分：{sum(checks)}/8")
if all(checks):
    print("通过：你已经走通语料→ARPA→G.fst 的完整链路。")
else:
    print("回到对应实验，用 ARPA 行、fstprint 和数值对照找证据。")


<details><summary>完成后展开参考答案</summary>

1. `log10`；2. `-ln(p)`；3. `backoff`；4. `<unk>`；5. `#0`；6. `lower`；7. `validation`；8. `arpa2fst`。

</details>

## 离场票

不看上文，写出 `lmplz` 训练命令、ARPA 一条 unigram/bigram/trigram 的列含义，以及 `arpa2fst` 转换命令。解释为什么 ARPA 的 log10 值不能原样当作 OpenFst 的自然对数代价。

下一课：[词典 L、消歧符号与 HCLG / CTC-TLG](语言模型零基础_06_词典L消歧与HCLG_CTC_TLG.ipynb)：把 `L` 的 disambiguation loop 与 `G.#0` backoff 正确连接，并构造、排错真实 `L∘G`。
